In [ ]:
import polars as pl
import numpy as np
import pickle
from datasets import load_dataset

import tensorflow as tf
import tensorflow_hub as hub

import spacy

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (f1_score, fbeta_score, hamming_loss,
                              jaccard_score, classification_report, accuracy_score)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df_train = pl.read_parquet('../../../EDA/df_train.parquet')
df_test = pl.read_parquet('../../../EDA/df_test.parquet')
df_validation = pl.read_parquet('../../../EDA/df_val.parquet')

In [ ]:
def to_polars_filtered(df: pl.DataFrame) -> pl.DataFrame:
    return df.filter(
        ~(
            pl.col("labels").list.contains(1) |
            pl.col("labels").list.contains(5) |
            pl.col("labels").list.contains(6)
        )
    )

df_train      = to_polars_filtered(df_train)
df_test       = to_polars_filtered(df_test)
df_validation = to_polars_filtered(df_validation)

In [ ]:
nlp = spacy.load("en_core_web_lg")

def clean(text, nlp):
    doc = nlp(text.lower().strip())
    return " ".join([token.text for token in doc])

def clean_series(df):
    docs = df["text"].to_list()
    cleaned = [clean(d, nlp) for d in docs]
    return df.with_columns(pl.Series("clean-text", cleaned))

df_train      = clean_series(df_train)
df_test       = clean_series(df_test)
df_validation = clean_series(df_validation)
print("Cleaning done.")

In [ ]:
elmo_model = hub.load("https://tfhub.dev/google/elmo/3").signatures["default"]

def elmo_vectors(texts):
    embeddings = elmo_model(tf.constant(list(texts)))["elmo"]
    return tf.reduce_mean(embeddings, axis=1).numpy()

def embed_dataframe(df, batch_size=100):
    batches = [df[i:i+batch_size] for i in range(0, df.shape[0], batch_size)]
    return np.concatenate([elmo_vectors(b["clean-text"]) for b in batches], axis=0)

X_train = embed_dataframe(df_train)
X_test  = embed_dataframe(df_test)
X_val   = embed_dataframe(df_validation)

# Save to disk (optional, avoids re-computing on restart)
for name, arr in [("elmo-train", X_train), ("elmo-test", X_test), ("elmo-val", X_val)]:
    with open(f"{name}.pkl", "wb") as f:
        pickle.dump(arr, f)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape} | X_val: {X_val.shape}") 

In [ ]:
y_train_raw = df_train["labels"].to_list()
y_test_raw  = df_test["labels"].to_list()
y_val_raw   = df_validation["labels"].to_list()

all_labels   = y_train_raw + y_test_raw + y_val_raw
all_label_ids = sorted(set(item for sub in all_labels for item in sub))

mlb = MultiLabelBinarizer(classes=all_label_ids)
mlb.fit(all_labels)

y_train = mlb.transform(y_train_raw)
y_test  = mlb.transform(y_test_raw)
y_val   = mlb.transform(y_val_raw)

label_dict = {0: "joy", 2: "fear", 3: "surprise", 4: "sadness", 7: "anger", 8: "disgust"}
label_names = [label_dict[c] for c in mlb.classes_]
print("Labels:", label_names)

In [ ]:
def find_best_thresholds(y_true, y_proba, num_labels, beta=1.0):
    """Per-label threshold search on F-beta score (beta=1 → F1, beta=0.5 → penalise FP)."""
    best_thresholds = []
    for i in range(num_labels):
        thresholds = np.arange(0.1, 0.9, 0.05)
        best_score, best_th = 0, 0.5
        for th in thresholds:
            y_pred_i = (y_proba[:, i] >= th).astype(int)
            score = fbeta_score(y_true[:, i], y_pred_i, beta=beta, zero_division=0)
            if score > best_score:
                best_score, best_th = score, th
        best_thresholds.append(best_th)
    return best_thresholds


def apply_thresholds(y_proba, thresholds):
    y_pred = np.zeros_like(y_proba)
    for i, th in enumerate(thresholds):
        y_pred[:, i] = (y_proba[:, i] >= th).astype(int)
    return y_pred


def evaluate(y_true, y_pred, label_names, title=""):
    print(f"
{'='*55}")
    print(f"  {title}")
    print(f"{'='*55}")
    print(f"Hamming Loss   : {hamming_loss(y_true, y_pred):.4f}  (↓ better)")
    print(f"Jaccard (macro): {jaccard_score(y_true, y_pred, average='macro', zero_division=0):.4f}  (↑ better)")
    print(f"F1 (macro)     : {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}  (↑ better)")
    print(f"Subset Accuracy:    {accuracy_score(y_test, y_pred):.4f}")
    print()
    print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))


NUM_LABELS = y_train.shape[1]
print("Helpers ready. NUM_LABELS:", NUM_LABELS)

In [ ]:
# ── Base ────────────────────────────────────────────
lgbm_base = OneVsRestClassifier(LGBMClassifier(n_estimators=100, class_weight="balanced", random_state=42, verbose=-1))
lgbm_base.fit(X_train, y_train)

ths = find_best_thresholds(y_val, lgbm_base.predict_proba(X_val), NUM_LABELS)
evaluate(y_test, apply_thresholds(lgbm_base.predict_proba(X_test), ths), label_names, "LGBM — Base")

In [ ]:
# ── GridSearch ──────────────────────────────────────
param_grid_lgbm = {
    "estimator__n_estimators":  [100, 200],
    "estimator__learning_rate": [0.05, 0.1, 0.2],
    "estimator__num_leaves":    [31, 63],
    "estimator__max_depth":     [-1, 10],
    "estimator__class_weight":  ["balanced", None],
}
grid_lgbm = GridSearchCV(
    OneVsRestClassifier(LGBMClassifier(random_state=42, verbose=-1)),
    param_grid_lgbm, cv=3, scoring="f1_macro", n_jobs=-1
)
grid_lgbm.fit(X_train, y_train)
print("Best:", grid_lgbm.best_params_)

ths = find_best_thresholds(y_val, grid_lgbm.best_estimator_.predict_proba(X_val), NUM_LABELS)
evaluate(y_test, apply_thresholds(grid_lgbm.best_estimator_.predict_proba(X_test), ths), label_names, "LGBM — GridSearch")

In [ ]:
# ── Optuna ──────────────────────────────────────────
def objective_lgbm(trial):
    n_est      = trial.suggest_int("n_estimators", 50, 400)
    lr         = trial.suggest_float("learning_rate", 0.01, 0.3, log=True)
    num_leaves = trial.suggest_int("num_leaves", 20, 150)
    max_d      = trial.suggest_int("max_depth", 3, 15)
    reg_alpha  = trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True)
    reg_lambda = trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True)
    cw         = trial.suggest_categorical("class_weight", ["balanced", None])
    clf = OneVsRestClassifier(LGBMClassifier(
        n_estimators=n_est, learning_rate=lr, num_leaves=num_leaves, max_depth=max_d,
        reg_alpha=reg_alpha, reg_lambda=reg_lambda, class_weight=cw,
        random_state=42, verbose=-1))
    clf.fit(X_train, y_train)
    return jaccard_score(y_val, clf.predict(X_val), average="macro", zero_division=0)

study_lgbm = optuna.create_study(direction="maximize")
study_lgbm.optimize(objective_lgbm, n_trials=40)
print("Best:", study_lgbm.best_params)

p = study_lgbm.best_params
lgbm_opt = OneVsRestClassifier(LGBMClassifier(
    n_estimators=p["n_estimators"], learning_rate=p["learning_rate"],
    num_leaves=p["num_leaves"], max_depth=p["max_depth"],
    reg_alpha=p["reg_alpha"], reg_lambda=p["reg_lambda"],
    class_weight=p["class_weight"], random_state=42, verbose=-1))
lgbm_opt.fit(X_train, y_train)

ths = find_best_thresholds(y_val, lgbm_opt.predict_proba(X_val), NUM_LABELS)
evaluate(y_test, apply_thresholds(lgbm_opt.predict_proba(X_test), ths), label_names, "LGBM — Optuna")